# Ring example

Reproduces the ring example results of Ashoff (2026), *Persistent Convolution: A Topological Approach to
Formal AI Alignment Testing* (DOI: [10.18130/8k9j-9k42](https://doi.org/10.18130/8k9j-9k42)):
mean representations with 95% pointwise confidence bounds
(top), energy statistics against the two-ring baseline with significance stars (middle), and
Wasserstein / JS distance heatmaps (bottom).

Data is synthetic and generated in-notebook with a fixed seed. The reproduced figure is
statistically identical to the published one but not pixel-identical (different bootstrap draws
than the original pipeline).

In [ ]:
import numpy as np
import persiscope as ps

rng = np.random.default_rng(0)


def rings(n_rings, n_per, noise, dim=6):
    """Disjoint noisy circles embedded in `dim` dimensions."""
    pts = []
    for r in range(n_rings):
        theta = rng.uniform(0, 2 * np.pi, n_per)
        center = np.array([r * 6.0, 0.0])
        ring2d = np.column_stack([np.cos(theta), np.sin(theta)]) * 2 + center
        ring2d += rng.normal(0, noise, size=ring2d.shape)
        pad = rng.normal(0, noise, size=(n_per, dim - 2))
        pts.append(np.hstack([ring2d, pad]))
    return np.vstack(pts)


two_rings = rings(2, 30, 0.10)      # the baseline
three_rings = rings(3, 20, 0.10)    # different component structure
noisy_rings = rings(2, 30, 0.90)    # same structure, heavy noise

In [ ]:
tf = ps.TopologicalTransformer(
    homology_dim=0,
    theta=-3 * np.pi / 8,   # the dissertation's diagram rotation
    n_bootstrap=50,
    random_state=0,
)
reps = [
    tf.fit_transform(x, label=lab)
    for x, lab in [(two_rings, "2 rings"), (three_rings, "3 rings"), (noisy_rings, "noisy rings")]
]

In [ ]:
fig = ps.viz.plot_baseline_report(
    reps,
    baseline=0,                          # everything is compared to "2 rings"
    curve_metrics=("wasserstein", "js"),
    n_permutations=200,
    random_state=0,
)

Expected reading, matching the published figure: `3 rings` differs significantly from the
baseline in every panel (different H0 structure), `noisy rings` shows a smaller but still
detectable separation, and the baseline column is `ns` by construction.